In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split

In [3]:
# Load dataset
df = pd.read_csv(r"D:\LLM\Spam_SMS.csv", encoding='latin-1')
df.columns = ['label', 'message']

# Convert labels: ham → 0, spam → 1
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

# Check data
print(df.head())

   label                                            message
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...


In [5]:
def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r'http\S+|www\S+', '', text)  # remove URLs
    text = re.sub(r'\d+', '', text)  # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = text.strip()  # remove spaces
    return text

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df['message'],
    df['label'],
    test_size=0.2,
    random_state=42
)

In [9]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [10]:
max_len = 100

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

In [11]:
model = Sequential()

model.add(Embedding(input_dim=5000, output_dim=64, input_length=max_len))
model.add(LSTM(64))
model.add(Dense(1, activation='sigmoid'))

In [13]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [15]:
model.fit(
    X_train_pad,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/20
112/112 [==============================] - 2s 14ms/step - loss: 0.0013 - accuracy: 0.9997 - val_loss: 0.0777 - val_accuracy: 0.9821
Epoch 2/20
112/112 [==============================] - 2s 13ms/step - loss: 5.9722e-04 - accuracy: 1.0000 - val_loss: 0.0883 - val_accuracy: 0.9798
Epoch 3/20
112/112 [==============================] - 2s 14ms/step - loss: 4.4337e-04 - accuracy: 1.0000 - val_loss: 0.0924 - val_accuracy: 0.9798
Epoch 4/20
112/112 [==============================] - 2s 14ms/step - loss: 3.2454e-04 - accuracy: 1.0000 - val_loss: 0.1008 - val_accuracy: 0.9798
Epoch 5/20
112/112 [==============================] - 2s 14ms/step - loss: 2.4996e-04 - accuracy: 1.0000 - val_loss: 0.0948 - val_accuracy: 0.9809
Epoch 6/20
112/112 [==============================] - 2s 14ms/step - loss: 1.9870e-04 - accuracy: 1.0000 - val_loss: 0.1074 - val_accuracy: 0.9809
Epoch 7/20
112/112 [==============================] - 2s 14ms/step - loss: 1.6682e-04 - accuracy: 1.0000 - val_loss: 0.106

In [16]:
loss, accuracy = model.evaluate(X_test_pad, y_test)

print("Accuracy:", accuracy)

35/35 [==============================] - 0s 6ms/step - loss: 0.1108 - accuracy: 0.9848
Accuracy: 0.9847533702850342


In [22]:
sample = ["Congratulations! You won a free prize"]

sample_seq = tokenizer.texts_to_sequences(sample)
sample_pad = pad_sequences(sample_seq, maxlen=max_len)

prediction = model.predict(sample_pad)

print("Spam" if prediction[0] > 0.5 else "Not Spam")

1/1 [==============================] - 0s 32ms/step
Not Spam


In [18]:
from tensorflow.keras.layers import Dropout, Bidirectional

model = Sequential()
model.add(Embedding(5000, 64, input_length=max_len))
model.add(Bidirectional(LSTM(64)))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))